## Human Message

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# env_path = Path('D:/.env')
# load_dotenv(dotenv_path=env_path)  # Load environment variables from the .env file

load_dotenv()

In [3]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users (metadata)
    id="msg_123",  # Optional: unique identifier for tracing (metadata)
    additional_kwargs={"source": "email", "priority": "high"}  # Optional: extra metadata 
)
human_msg

HumanMessage(content='Hello!', additional_kwargs={'source': 'email', 'priority': 'high'}, response_metadata={}, name='alice', id='msg_123')

In [2]:
from langchain.messages import HumanMessage

human_msg_1 = HumanMessage(
    content="Hello!")
human_msg_1

HumanMessage(content='Hello!', additional_kwargs={}, response_metadata={})

In [9]:
type(human_msg)  # Output: <class 'langchain.messages.HumanMessage'>

langchain_core.messages.human.HumanMessage

In [21]:
# Display the attributes of the HumanMessage instance
print(human_msg.content)                    # Hello!
print(human_msg.name)                       # alice
print(human_msg.id)                         # msg_123
print(human_msg.additional_kwargs)          # {'source': 'email', 'priority': 'high'}

Hello!
alice
msg_123
{'source': 'email', 'priority': 'high'}


## 客戶訊息分類的訊息提示詞

In [3]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(
        content=(
            "You classify customer messages as Billing, Delivery, or Product. "
            "Return only the category name."
        )
    ),
    HumanMessage(
        content="The courier marked my order as delivered, but I did not receive it."
    )
]

# Create a model
model = init_chat_model("gpt-5-nano")

# Invoke the model with the messages
response = model.invoke(messages)
response

AIMessage(content='Delivery', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 43, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQ6mMzUWytCrhPHkg4DXsPDUiFGTL', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0bdd5-ae89-7142-a854-3b8d0d85d72b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 74, 'total_tokens': 117, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 64}})

## Prompt Template

In [1]:
prompt_template_str = """
  {task_instruction}
  Relevant examples:
  {selected_examples}
  New message: {user_input}
  Category:
"""

In [3]:
from langchain_core.prompts import PromptTemplate
# Create a PromptTemplate from the f-string template
prompt_template = PromptTemplate.from_template(prompt_template_str)

In [4]:
# Define the runtime values for the placeholders
task_instruction = "Classify each customer message as Billing, Delivery, or Product. Return only the category name."

selected_examples = """
Message: My parcel has not arrived yet.
Category: Delivery

Message: The tracking page says my parcel was sent to the wrong city.
Category: Delivery
"""

user_input = "The courier marked my order as delivered, but I did not receive it."
# Assign values to placeholders and render the prompt
prompt = prompt_template.invoke({
    "task_instruction": task_instruction,
    "selected_examples": selected_examples,
    "user_input": user_input
})

# View the rendered prompt
from pprint import pprint
pprint(prompt)

StringPromptValue(text='\n  Classify each customer message as Billing, Delivery, or Product. Return only the category name.\n  Relevant examples:\n  \nMessage: My parcel has not arrived yet.\nCategory: Delivery\n\nMessage: The tracking page says my parcel was sent to the wrong city.\nCategory: Delivery\n\n  New message: The courier marked my order as delivered, but I did not receive it.\n  Category:\n')


## Chat Message Template

In [5]:
messages_template = [
    ("system", "You classify customer messages as Billing, Delivery, or Product. Return only the category name."),
    ("user", "Relevant examples:\n{selected_examples}"),
    ("user", "New message: {user_input}\nCategory:")
]

In [7]:
from langchain_core.prompts import ChatPromptTemplate
# Create a ChatPromptTemplate from the message list
chat_prompt_template = ChatPromptTemplate.from_messages(messages_template)

In [8]:
prompt = chat_prompt_template.invoke({
    "selected_examples": selected_examples,
    "user_input": user_input
})

In [ ]:
pprint(prompt)

ChatPromptValue(messages=[SystemMessage(content='You classify customer messages as Billing, Delivery, or Product. Return only the category name.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Relevant examples:\n\nMessage: My parcel has not arrived yet.\nCategory: Delivery\n\nMessage: The tracking page says my parcel was sent to the wrong city.\nCategory: Delivery\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='New message: The courier marked my order as delivered, but I did not receive it.\nCategory:', additional_kwargs={}, response_metadata={})])


## 強迫 JSON 格式輸出

In [11]:
from typing import Literal
from pydantic import BaseModel, Field

class CustomerMessageClassification(BaseModel):
    """The classification result for a customer message."""

    category: Literal["Billing", "Delivery", "Product"] = Field(
        description="The category assigned to the customer message."
    )
    reason: str = Field(
        description="A short explanation for the classification."
    )

In [12]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

structured_model = model.with_structured_output(
    CustomerMessageClassification
)

In [13]:
messages = [
    {
        "role": "system",
        "content": (
            "Classify each customer message as Billing, Delivery, or Product. "
            "Provide a short reason for the classification."
        )
    },
    {
        "role": "user",
        "content": (
            "The courier marked my order as delivered, "
            "but I did not receive it."
        )
    }
]



In [23]:
result = structured_model.invoke(messages)



In [25]:

response: CustomerMessageClassification = CustomerMessageClassification.model_validate(result)
type(response)

__main__.CustomerMessageClassification

In [26]:
print(response)
print(response.category)
print(response.reason)

category='Delivery' reason='Delivery issue: package marked as delivered but not received.'
Delivery
Delivery issue: package marked as delivered but not received.
